<a href="https://colab.research.google.com/github/0sinach1/house-price-model/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Price Prediction in Nigeria

This notebook covers exploratory data analysis, preprocessing, modeling, and evaluation for predicting house prices in Nigeria.


## 1. Import Required Libraries

We will use pandas, numpy, matplotlib, seaborn, and scikit-learn for data analysis and modeling.

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Load and Explore the Dataset

Load the Nigeria house prices dataset and display basic information.

In [8]:
# Load the dataset
file_path = "/content/drive/MyDrive/Datasets/nigeria_houses_data.csv"
df = pd.read_csv(file_path)
df.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
0,6.0,5.0,5.0,4.0,Detached Duplex,Mabushi,Abuja,450000000.0
1,4.0,5.0,5.0,4.0,Terraced Duplexes,Katampe,Abuja,800000000.0
2,4.0,5.0,5.0,4.0,Detached Duplex,Lekki,Lagos,120000000.0
3,4.0,4.0,5.0,6.0,Detached Duplex,Ajah,Lagos,40000000.0
4,4.0,4.0,5.0,2.0,Semi Detached Duplex,Lekki,Lagos,75000000.0


In [9]:

print("Creating new features...")

# 1. Price per square meter (if you have sqm column)
# Check if 'sqm' or 'size' or 'area' column exists
if 'sqm' in df.columns:
    df['price_per_sqm'] = df['price'] / df['sqm']
    print("✅ Created: price_per_sqm")
elif 'size' in df.columns:
    df['price_per_sqm'] = df['price'] / df['size']
    print("✅ Created: price_per_sqm")
else:
    print("⚠️  No sqm/size column found - skipping price_per_sqm")

# 2. Bedroom to bathroom ratio
df['bedroom_bathroom_ratio'] = df['bedrooms'] / (df['bathrooms'] + 0.1)  # +0.1 to avoid division by zero
print("✅ Created: bedroom_bathroom_ratio")

# 3. Total rooms (bedrooms + bathrooms + toilets)
df['total_rooms'] = df['bedrooms'] + df['bathrooms'] + df['toilets']
print("✅ Created: total_rooms")

# 4. Luxury score (sum of premium features)
df['luxury_score'] = df['parking_space'].clip(upper=5)  # Cap at 5
print("✅ Created: luxury_score")

# 5. Is Lagos (binary feature - Lagos properties are typically more expensive)
df['is_lagos'] = (df['state'] == 'Lagos').astype(int)
print("✅ Created: is_lagos")

# 6. Is Abuja (second most expensive)
df['is_abuja'] = (df['state'] == 'Abuja').astype(int)
print("✅ Created: is_abuja")

# 7. Is premium location (Lagos or Abuja)
df['is_premium_location'] = ((df['state'] == 'Lagos') | (df['state'] == 'Abuja')).astype(int)
print("✅ Created: is_premium_location")

# Display new features
print("\n" + "="*60)
print("NEW FEATURES CREATED")
print("="*60)
print(df[['bedrooms', 'bathrooms', 'bedroom_bathroom_ratio',
          'total_rooms', 'luxury_score', 'is_lagos', 'is_abuja']].head())
print("\n✅ Feature engineering complete!")

Creating new features...
⚠️  No sqm/size column found - skipping price_per_sqm
✅ Created: bedroom_bathroom_ratio
✅ Created: total_rooms
✅ Created: luxury_score
✅ Created: is_lagos
✅ Created: is_abuja
✅ Created: is_premium_location

NEW FEATURES CREATED
   bedrooms  bathrooms  bedroom_bathroom_ratio  total_rooms  luxury_score  \
0       6.0        5.0                1.176471         16.0           4.0   
1       4.0        5.0                0.784314         14.0           4.0   
2       4.0        5.0                0.784314         14.0           4.0   
3       4.0        4.0                0.975610         13.0           5.0   
4       4.0        4.0                0.975610         13.0           2.0   

   is_lagos  is_abuja  
0         0         1  
1         0         1  
2         1         0  
3         1         0  
4         1         0  

✅ Feature engineering complete!


### 2.1 Data Structure and Missing Values

Check the dataframe's shape, info, and count missing values.

In [ ]:
# DataFrame info and shape
print("Shape:", df.shape)
df.info()

In [ ]:
# Count missing values for each column
df.isnull().sum()

### 2.2 Visualize Target Variable Distribution

Plot the distribution of the 'price' column and its log-transformed version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['price'], bins=30, kde=True)
plt.title('Original Price Distribution')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['price']), bins=30, kde=True)
plt.title('Log-Transformed Price Distribution')
plt.xlabel('log(Price)')
plt.tight_layout()
plt.show()

### 2.3 Analyze Categorical Features

Display the most common states in the dataset.

In [ ]:
# Top 10 states by count
state_counts = df['state'].value_counts().head(10)
print(state_counts)

## 3. Data Preprocessing

Prepare data for modeling: define features and target, then split into train/test sets.

In [10]:
numeric_features = [
    'bedrooms', 'bathrooms', 'toilets', 'parking_space',
    'bedroom_bathroom_ratio', 'total_rooms', 'luxury_score'
]

# Add price_per_sqm if it was created
if 'price_per_sqm' in df.columns:
    numeric_features.append('price_per_sqm')

# Categorical features
categorical_features = ['state', 'town']

# Target variable
target = 'price'

print("="*60)
print("FEATURES SELECTED")
print("="*60)
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Target: {target}")
print("="*60)

# Prepare X and y
X = df[numeric_features + categorical_features]
y = df[target]

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

FEATURES SELECTED
Numeric features (7): ['bedrooms', 'bathrooms', 'toilets', 'parking_space', 'bedroom_bathroom_ratio', 'total_rooms', 'luxury_score']
Categorical features (2): ['state', 'town']
Target: price

X shape: (24326, 9)
y shape: (24326,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train set:", X_train.shape)
print("Test set:", X_test.shape)


Train set: (19460, 9)
Test set: (4866, 9)
